# Chicago Parking Ticket Data Analysis

We will use a notebook to perform some basic analysis of the Chicago Parking Ticket data sample.

This is a 1 million row randomized sample from [Daniel Hutmacher's Chicago Parking Tickets database](https://sqlsunday.com/2022/12/05/new-demo-database/).  I have added a new label column, `PaymentIsOutstanding`, which represents whether the ticket recipient still owes the City of Chicago any money.

Our goal in the subsequent analysis is to gain a better feel for this dataset and some of the things we will need to do when we want to run an experiment to train a model.

## Retrieve Data

In [6]:
# from azureml.core import Workspace, Datastore, Dataset
import numpy as np
import pandas as pd

# ws=Workspace.from_config()
# ws

In [ ]:
#### Error
#### Request failed with: 404 Client Error: User error when calling GenericAssetMLIndexServiceClient.MoveNext. Service invocation failed!Request for url: https://westus.experiments.azureml.net/data/v1.0/subscriptions/d2b4c44c-e208-4a11-b2e3-64224655d213/resourceGroups/aml-rg/providers/Microsoft.MachineLearningServices/workspaces/xp-aml/dataversion/ChicagoParkingTickets/versions/None?includeSavedDatasets=true
#### Tried to retrieve v2 data asset but could not find v2 data assetregistered with name "ChicagoParkingTickets" (version: None) in the workspace.

# dataset=Dataset.get_by_name(ws, name='ChicagoParkingTickets')
# df=dataset.to_pandas_dataframe()

In [ ]:
#### error
#### Exception: StreamError(PermissionDenied(Some(This request is not authorized to perform this operation using this permission.)))
# import pandas as pd
# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential

# ml_client = MLClient.from_config(credential=DefaultAzureCredential())
# data_asset = ml_client.data.get("xpdatalake_chicago_csv", version="1")

# df = pd.read_csv(data_asset.path)
# df

In [ ]:
#### copied from AML Studio
#### Data > Data assets > ChicagoParkingTicketsFolder > Consume
import mltable
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get("ChicagoParkingTicketsFolder", version="1")

path = {
  'folder': data_asset.path
}

tbl = mltable.from_delimited_files(paths=[path])
df = tbl.to_pandas_dataframe()
df

## Perform Exploratory Analysis

`len()` tells us how many rows there are in a dataframe.

In [2]:
len(df)

1000000

Review the top few rows in the dataframe.

In [3]:
df.head()

,Issued_date,Community_Name,Sector,Side,Hardship_Index,Per_capita_income,Percent_unemployed,Percent_without_diploma,Percent_households_below_poverty,Neighborhood,Ward,Tract,ZIP,Police_District,Plate_Type,License_Plate_State,Unit_ID,Violation_ID,PaymentIsOutstanding
0,2000-01-29 09:10:00,Austin,Other W Side,West Side,73.0,15957.0,22.6,24.4,28.6,A3,29,252101.0,60644,15.0,PAS,IL,15,123,True
1,2014-08-21 09:31:00,Chatham,Other S/SE/SW Side,Far Southeast Side,60.0,18881.0,24.0,14.5,27.8,C1,8,440102.0,60619,6.0,PAS,IL,498,6,False
2,2015-04-10 19:41:00,West Town,West Town,West Side,10.0,43198.0,6.6,12.9,14.7,WP3,1,241400.0,60622,14.0,PAS,IL,502,44,False
3,2014-10-10 00:32:00,Hermosa,Other N/NW Side,Northwest Side,71.0,15089.0,13.1,41.6,20.5,H2,31,200100.0,60641,25.0,PAS,IL,502,163,False
4,2015-12-19 15:46:00,Washington Heights,Other S/SE/SW Side,Far Southwest Side,48.0,19713.0,20.8,13.7,16.9,WH,21,730201.0,60620,22.0,PAS,IL,22,169,False


We have a mix of datetime, string, and numeric data, as well as one label:  `PaymentIsOutstanding`.

What is the set of unique values for one of these columns?  I'll choose `Police_District` as an example.

In [4]:
df['Police_District'].unique()

array([15.,  6., 14., 25., 22., 16.,  9., 19., 12.,  8.,  1., 20.,  2.,
       18., 10., 11.,  3.,  7., 24., 17.,  4.,  5., nan])

It looks like police district is really an integer, even though it's saved as a decimal.  Also, `NaN` appears, which indicates some police districts were either missing or had data which didn't fit the datatype.  We'll want to get back to this.

Before we go too much further, what is the cardinality (distinct count) of every column?

In [7]:
df.apply(pd.Series.nunique)

Issued_date                         914054
Community_Name                          77
Sector                                  13
Side                                     9
Hardship_Index                          77
Per_capita_income                       77
Percent_unemployed                      66
Percent_without_diploma                 69
Percent_households_below_poverty        67
Neighborhood                            98
Ward                                    50
Tract                                  799
ZIP                                     59
Police_District                         22
Plate_Type                              53
License_Plate_State                     64
Unit_ID                                160
Violation_ID                           157
PaymentIsOutstanding                     2
dtype: int64

We can see that issued date is almost entirely unique.  For analysis, we might want to split this either into smaller components (e.g., year, month, day columns) or split out the date and time.  We may also wish to bin the times in some fashion, such as into 6-hour blocks.

We can also pick out a little bit more from this as well:

* Every community has a few measures: hardship index, per-capita income, percent unemployed, percent without diploma, percent households below poverty.  For the most part, these columns are unique.
* Given that there are 22 police districts, 13 sectors, 98 neighborhoods, 77 communities, and 59 ZIP codes, there's going to be some overlap between these.
* There are 64 license plate states, which may seem weird when you consider there are 50 states in the US.  This dataset also includes out-of-country visitors, especially from Canada.  It might make sense to reshape this data:  in-state, out-of-state, out-of-country.

In [8]:
pd.options.display.max_rows=64
df['License_Plate_State'].value_counts()

IL    914922
IN     17695
MI     11064
WI     10344
IA      5485
OH      5178
TX      3535
CA      3064
FL      2961
MN      2692
MO      2625
NY      1353
PA      1343
GA      1312
TN      1309
AZ      1079
KY      1034
CO       938
VA       915
NC       861
NJ       792
MD       756
MA       690
WA       683
KS       669
OK       623
AL       519
MS       514
LA       407
CT       399
AR       386
SC       382
NE       380
OR       348
ON       343
NV       266
NM       222
ZZ       200
UT       166
NH       132
SD       128
MT       117
ND       114
ME       111
VT        95
WV        91
RI        78
AK        75
DC        74
WY        71
NB        69
ID        68
DE        45
AB        42
QU        41
HI        37
VI        35
MX        30
BC        16
MB        13
PE         8
NS         4
PQ         4
NF         1
Name: License_Plate_State, dtype: int64

`Plate_Type` is intended to indicate what kind of vehicle this is.

In [9]:
pd.options.display.max_rows=53
df['Plate_Type'].value_counts()

PAS    871404
TRK     51936
TMP     41863
DLC      4331
HCP      3895
ENV      3242
MCY      2626
TXI      2412
RVM      1520
LIV      1037
APP       823
COL       777
VAN       333
TOW       273
SPM       251
ANV       229
INT       199
PFR       174
SCH       161
CHV       117
ELE       105
PTR        99
TRL        92
BUS        67
MUN        67
EDU        59
TRN        58
GRD        56
MFR        48
AFR        43
ANM        30
TRT        29
RVT        22
HCS        18
FHM        17
STE        16
AMB        15
FRM        11
DLM        10
FCH         8
ICC         8
RPO         8
IMC         7
DLT         5
SPV         4
SPC         4
SOS         3
CON         2
EXP         2
SHF         2
HIM         1
FER         1
DRV         1
Name: Plate_Type, dtype: int64

We can see that after the first three plate types, there are very few examples of any other.  This indicates that we probably want to have Passenger (PAS), Truck (TRK), Temporary Tags (TMP), and Other.

Let's take a look at some of the rows which are marked as having outstanding payments.

In [10]:
df[df['PaymentIsOutstanding'] == 1].head()

,Issued_date,Community_Name,Sector,Side,Hardship_Index,Per_capita_income,Percent_unemployed,Percent_without_diploma,Percent_households_below_poverty,Neighborhood,Ward,Tract,ZIP,Police_District,Plate_Type,License_Plate_State,Unit_ID,Violation_ID,PaymentIsOutstanding
0,2000-01-29 09:10:00,Austin,Other W Side,West Side,73.0,15957.0,22.6,24.4,28.6,A3,29,252101.0,60644,15.0,PAS,IL,15,123,True
13,2017-09-27 15:11:00,Near West Side,Near West Side,West Side,15.0,44689.0,10.7,9.6,20.6,LI,27,838200.0,60612,12.0,PAS,IL,498,113,True
14,2002-03-14 09:45:00,Archer Heights,Other S/SE/SW Side,Southwest Side,67.0,16134.0,16.5,35.9,14.1,AH,14,570300.0,60632,8.0,PAS,IL,498,123,True
17,2005-04-02 20:03:00,Logan Square,Logan Square,North Side,23.0,31908.0,8.2,14.8,16.8,B4,32,830900.0,60647,14.0,PAS,IL,14,123,True
20,2011-10-30 22:30:00,Montclare,Other N/NW Side,Northwest Side,50.0,22014.0,13.8,23.5,15.3,M,29,831600.0,60707,25.0,PAS,IL,25,177,True


And here are a few rows where payment is complete.  This could be because the person made payment or because a judge threw out the ticket.

In [11]:
df[df['PaymentIsOutstanding'] == 0].head()

,Issued_date,Community_Name,Sector,Side,Hardship_Index,Per_capita_income,Percent_unemployed,Percent_without_diploma,Percent_households_below_poverty,Neighborhood,Ward,Tract,ZIP,Police_District,Plate_Type,License_Plate_State,Unit_ID,Violation_ID,PaymentIsOutstanding
1,2014-08-21 09:31:00,Chatham,Other S/SE/SW Side,Far Southeast Side,60.0,18881.0,24.0,14.5,27.8,C1,8,440102.0,60619,6.0,PAS,IL,498,6,False
2,2015-04-10 19:41:00,West Town,West Town,West Side,10.0,43198.0,6.6,12.9,14.7,WP3,1,241400.0,60622,14.0,PAS,IL,502,44,False
3,2014-10-10 00:32:00,Hermosa,Other N/NW Side,Northwest Side,71.0,15089.0,13.1,41.6,20.5,H2,31,200100.0,60641,25.0,PAS,IL,502,163,False
4,2015-12-19 15:46:00,Washington Heights,Other S/SE/SW Side,Far Southwest Side,48.0,19713.0,20.8,13.7,16.9,WH,21,730201.0,60620,22.0,PAS,IL,22,169,False
5,2017-07-03 11:14:00,Portage Park,Other N/NW Side,Northwest Side,35.0,24336.0,12.6,19.3,11.6,PP,30,151100.0,60641,16.0,PAS,IL,16,10,False


## Correlation Analysis

Something we'll want to do is perform a basic correlation analysis.  We'll focus on the numeric features and see if any of these correlate to whether payment is outstanding.

In [12]:
pd.options.display.max_rows=20
df.corr(numeric_only=True)

,Hardship_Index,Per_capita_income,Percent_unemployed,Percent_without_diploma,Percent_households_below_poverty,Ward,Tract,ZIP,Police_District,Unit_ID,Violation_ID,PaymentIsOutstanding
Hardship_Index,1.000000,-0.873156,0.842290,0.896544,0.816899,-0.332984,0.342031,0.051665,-0.189453,-0.150071,0.007028,0.138923
Per_capita_income,-0.873156,1.000000,-0.733523,-0.794308,-0.677403,0.285006,-0.328733,-0.132169,0.159888,0.149546,0.002359,-0.123757
Percent_unemployed,0.842290,-0.733523,1.000000,0.570578,0.815885,-0.418000,0.392479,0.077121,-0.308673,-0.181943,0.019360,0.148347
Percent_without_diploma,0.896544,-0.794308,0.570578,1.000000,0.587978,-0.253830,0.263183,-0.005817,-0.087733,-0.085419,-0.004428,0.098042
Percent_households_below_poverty,0.816899,-0.677403,0.815885,0.587978,1.000000,-0.304524,0.344130,0.002619,-0.301773,-0.156410,0.006719,0.136414
Ward,-0.332984,0.285006,-0.418000,-0.253830,-0.304524,1.000000,-0.418862,0.189437,0.577929,0.041767,-0.011956,-0.060562
Tract,0.342031,-0.328733,0.392479,0.263183,0.344130,-0.418862,1.000000,-0.272146,-0.555396,-0.064678,0.013341,0.059178
ZIP,0.051665,-0.132169,0.077121,-0.005817,0.002619,0.189437,-0.272146,1.000000,0.352754,-0.000914,0.014768,0.005233
Police_District,-0.189453,0.159888,-0.308673,-0.087733,-0.301773,0.577929,-0.555396,0.352754,1.000000,0.048748,-0.017823,-0.058603
Unit_ID,-0.150071,0.149546,-0.181943,-0.085419,-0.156410,0.041767,-0.064678,-0.000914,0.048748,1.000000,-0.029990,-0.161045


As an initial look, there aren't many numeric features which correlate well linearly with `PaymentIsOutstanding`.

We can also see that some tight correlation between hardship index, per-capita income (in the negative direction), percent unemployed, percent without diploma, and percent households below poverty.  To avoid the risk of multicollinearity, we probably want to drop either hardship index or the other columns mentioned.

One last thing we'll do is include year and month features from the issuance date, and see if those are correlated at all with payment.

In [13]:
df['Year'], df['Month']=df['Issued_date'].dt.year, df['Issued_date'].dt.month
df.corr(numeric_only=True)

,Hardship_Index,Per_capita_income,Percent_unemployed,Percent_without_diploma,Percent_households_below_poverty,Ward,Tract,ZIP,Police_District,Unit_ID,Violation_ID,PaymentIsOutstanding,Year,Month
Hardship_Index,1.000000,-0.873156,0.842290,0.896544,0.816899,-0.332984,0.342031,0.051665,-0.189453,-0.150071,0.007028,0.138923,0.009509,-0.003528
Per_capita_income,-0.873156,1.000000,-0.733523,-0.794308,-0.677403,0.285006,-0.328733,-0.132169,0.159888,0.149546,0.002359,-0.123757,-0.012086,0.002261
Percent_unemployed,0.842290,-0.733523,1.000000,0.570578,0.815885,-0.418000,0.392479,0.077121,-0.308673,-0.181943,0.019360,0.148347,0.014905,-0.006097
Percent_without_diploma,0.896544,-0.794308,0.570578,1.000000,0.587978,-0.253830,0.263183,-0.005817,-0.087733,-0.085419,-0.004428,0.098042,0.007708,0.000283
Percent_households_below_poverty,0.816899,-0.677403,0.815885,0.587978,1.000000,-0.304524,0.344130,0.002619,-0.301773,-0.156410,0.006719,0.136414,-0.002007,-0.006847
Ward,-0.332984,0.285006,-0.418000,-0.253830,-0.304524,1.000000,-0.418862,0.189437,0.577929,0.041767,-0.011956,-0.060562,-0.032580,-0.001303
Tract,0.342031,-0.328733,0.392479,0.263183,0.344130,-0.418862,1.000000,-0.272146,-0.555396,-0.064678,0.013341,0.059178,0.007925,-0.003459
ZIP,0.051665,-0.132169,0.077121,-0.005817,0.002619,0.189437,-0.272146,1.000000,0.352754,-0.000914,0.014768,0.005233,0.015017,-0.001367
Police_District,-0.189453,0.159888,-0.308673,-0.087733,-0.301773,0.577929,-0.555396,0.352754,1.000000,0.048748,-0.017823,-0.058603,0.010619,0.003879
Unit_ID,-0.150071,0.149546,-0.181943,-0.085419,-0.156410,0.041767,-0.064678,-0.000914,0.048748,1.000000,-0.029990,-0.161045,0.378028,0.041632


It looks like the year has a small effect (older tickets are more likely to be paid up), but month of year doesn't have any real benefit to us.

## Final Cleanup

Before we wrap things up, we'll want to see what other types of data cleanup we might want to do.  For example, which columns have `NaN`?

In [14]:
pd.options.display.max_rows=30
df.isna().any()

Issued_date                         False
Community_Name                      False
Sector                              False
Side                                False
Hardship_Index                      False
Per_capita_income                   False
Percent_unemployed                  False
Percent_without_diploma             False
Percent_households_below_poverty    False
Neighborhood                        False
Ward                                False
Tract                                True
ZIP                                 False
Police_District                      True
Plate_Type                           True
License_Plate_State                  True
Unit_ID                             False
Violation_ID                        False
PaymentIsOutstanding                False
Year                                False
Month                               False
dtype: bool

It looks like the following columns do:

* Tract
* Police District
* Plate Type
* License Plate State

## Plan of Action

* Find and replace missing values for police district
* Create features for year, time block during the day, license plate origin (based on license plate state), vehicle type (based on plate type)
* Remove Hardship index and Census tract
* Encode categorical features so we can perform a classification analysis.